# Análisis vendedores RECESA

Este notebook transforma los archivos de ventas de **Allan** y **Fabiola** en una base maestra limpia para comparar desempeño por vendedor.

Archivos esperados en la misma carpeta del notebook:

- `Ventas redes RECESA Allan.xlsx`
- `Ventas redes RECESA Fabiola.xlsx`

Salida generada:

- `Base_Maestra_Vendedores_RECESA.xlsx`


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

## 1. Configuración de archivos

In [2]:
# =========================
# ARCHIVOS Y HOJAS
# =========================

archivo_allan = "Ventas redes RECESA Allan.xlsx"
archivo_fabiola = "Ventas redes RECESA Fabiola.xlsx"

hoja_allan = "Ventas_Recesa_Allan"
hoja_fabiola = "Ventas_Recesa_Fabiola"

archivos = {
    "Allan": {
        "archivo": archivo_allan,
        "hoja": hoja_allan
    },
    "Fabiola": {
        "archivo": archivo_fabiola,
        "hoja": hoja_fabiola
    }
}

for vendedor, info in archivos.items():
    if not Path(info["archivo"]).exists():
        raise FileNotFoundError(f"No encontré el archivo: {info['archivo']}")

print("Archivos encontrados correctamente")

Archivos encontrados correctamente


## 2. Leer archivos originales

In [3]:
df_allan_raw = pd.read_excel(
    archivo_allan,
    sheet_name=hoja_allan,
    header=None
)

df_fabiola_raw = pd.read_excel(
    archivo_fabiola,
    sheet_name=hoja_fabiola,
    header=None
)

print("Allan:", df_allan_raw.shape)
print("Fabiola:", df_fabiola_raw.shape)

df_allan_raw.head(8)

Allan: (879, 43)
Fabiola: (566, 46)


,0,1,2,3,4,5,6,7,8,9,...,33,34,35,36,37,38,39,40,41,42
0,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,mayo 2025,NaN,NaN,junio 2025,NaN,NaN,julio 2025,NaN,NaN,...,NaN,abril 2026,NaN,NaN,mayo 2026,NaN,NaN,NaN,NaN,NaN
2,NaN,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,...,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio
3,Total,134800.04,1071.5,112.325852,182507.56,2976,54.755773,175718.68,1899,82.618067,...,58.133853,202919.21,1356,133.614285,207562.18,1607,115.322589,2374813.9,28946.5,73.251449
4,PRODUCTO TERMINADO / BALANCIN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,607.16,2,271.055
5,[RL-2601-A] BALANCIN P/CARRETON ( JU...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,607.16,2,271.055
6,PRODUCTO TERMINADO / HOJAS,79830.97,227,313.998414,120069.56,356,301.137472,93671.46,291,287.406323,...,293.598405,137222.44,325,376.984708,126691.39,295,383.448712,1370399.3,3958,309.138717
7,[02011313-A] CARRETON 1a 13 3/8X13 3...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,909.22,4,202.95


In [4]:
df_fabiola_raw.head(8)

,0,1,2,3,4,5,6,7,8,9,...,36,37,38,39,40,41,42,43,44,45
0,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,mayo 2025,NaN,NaN,junio 2025,NaN,NaN,julio 2025,NaN,NaN,...,NaN,mayo 2026,NaN,NaN,junio 2026,NaN,NaN,NaN,NaN,NaN
2,NaN,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,...,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio
3,Total,955469.19,181385,4.703242,108852.11,17235,5.639073,591992.67,238130,2.219649,...,2.867847,150092.88,21026,6.373606,14038.2,1161,10.795952,3071117.34,853812,3.211561
4,PRODUCTO TERMINADO / HOJAS,546.61,1,488.04,3238.89,11,262.896364,NaN,NaN,NaN,...,NaN,2111.59,3,628.45,NaN,NaN,NaN,11463.9,91,112.479231
5,[01010325-M] CARRETON 1a 2.1/2X25 1....,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3456,72,42.857083
6,[21012229-B] TOYOTA HILUX 90 4WD TRA...,NaN,NaN,NaN,726.47,3,216.21,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,726.47,3,216.21
7,[25HL1417-A] TOYOTA HILUX AUXILIAR 5...,NaN,NaN,NaN,802.73,3,238.906667,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,802.73,3,238.906667


## 3. Funciones de limpieza y transformación

La estructura de estos Excel es:

- Fila 1: meses
- Fila 2: Total / Cantidad de producto / Precio promedio
- Fila 3: totales generales
- Fila 4 en adelante: categorías y productos

El código toma las filas con formato `[CODIGO] Descripción` como productos reales. Las filas como `PRODUCTO TERMINADO / HOJAS` se usan como categoría, pero no entran como producto.


In [5]:
def extraer_codigo_descripcion(texto):
    texto = str(texto).strip()

    match = re.search(r"\[(.*?)\]", texto)

    if match:
        codigo = match.group(1).strip()
        descripcion = re.sub(r"\[.*?\]", "", texto).strip()
        return pd.Series([codigo, descripcion])

    return pd.Series(["", texto])


def limpiar_categoria(texto):
    if pd.isna(texto):
        return None

    texto = str(texto).strip()
    texto_upper = texto.upper()

    if texto_upper.startswith("PRODUCTO TERMINADO"):
        if "/" in texto:
            return texto.split("/", 1)[1].strip()
        return texto.replace("PRODUCTO TERMINADO", "").strip()

    return None


def transformar_redes(df_raw, vendedor):
    df = df_raw.copy()

    producto_col = 0
    bloques = []

    # Los bloques de meses van cada 3 columnas: Total, Cantidad, Precio promedio
    for col in range(1, df.shape[1], 3):
        if col + 2 >= df.shape[1]:
            continue

        mes = str(df.iloc[1, col]).strip().lower()

        if mes in ["nan", "none", ""]:
            continue

        bloques.append({
            "mes": mes,
            "total": col,
            "cantidad": col + 1,
            "precio": col + 2
        })

    # Desde la fila 4 empiezan categorías y productos
    df_productos = df.iloc[4:].copy().reset_index(drop=True)

    # Detectar categoría y arrastrarla hacia abajo
    df_productos["Categoria_Row"] = df_productos[producto_col].apply(limpiar_categoria)
    df_productos["Categoria"] = df_productos["Categoria_Row"].ffill().fillna("SIN CATEGORIA")

    # Quedarse solo con filas que tienen código entre corchetes
    mask_producto = df_productos[producto_col].astype(str).str.contains(r"\[.*?\]", regex=True, na=False)
    df_productos = df_productos[mask_producto].copy().reset_index(drop=True)

    cod_desc = df_productos[producto_col].apply(extraer_codigo_descripcion)
    cod_desc.columns = ["Codigo", "Descripcion"]

    registros = []

    for bloque in bloques:
        temp = pd.DataFrame(index=df_productos.index)

        temp["Empresa"] = "RECESA"
        temp["Vendedor"] = vendedor
        temp["Categoria"] = df_productos["Categoria"]
        temp["Codigo"] = cod_desc["Codigo"]
        temp["Descripcion"] = cod_desc["Descripcion"]
        temp["Mes"] = bloque["mes"]

        temp["Total"] = pd.to_numeric(
            df_productos[bloque["total"]],
            errors="coerce"
        ).fillna(0)

        temp["Cantidad"] = pd.to_numeric(
            df_productos[bloque["cantidad"]],
            errors="coerce"
        ).fillna(0)

        temp["Precio_Promedio"] = pd.to_numeric(
            df_productos[bloque["precio"]],
            errors="coerce"
        ).fillna(0)

        registros.append(temp)

    df_final = pd.concat(registros, ignore_index=True)

    # Quitar filas sin movimiento en el mes
    df_final = df_final[
        (df_final["Total"] != 0) |
        (df_final["Cantidad"] != 0)
    ].reset_index(drop=True)

    return df_final

## 4. Transformar Allan y Fabiola

In [6]:
df_allan = transformar_redes(df_allan_raw, "Allan")
df_fabiola = transformar_redes(df_fabiola_raw, "Fabiola")

df_total_vendedores = pd.concat(
    [df_allan, df_fabiola],
    ignore_index=True
)

print("Filas Allan:", len(df_allan))
print("Filas Fabiola:", len(df_fabiola))
print("Filas totales:", len(df_total_vendedores))

df_total_vendedores.head(20)

Filas Allan: 2135
Filas Fabiola: 1286
Filas totales: 3421


,Empresa,Vendedor,Categoria,Codigo,Descripcion,Mes,Total,Cantidad,Precio_Promedio
0,RECESA,Allan,HOJAS,21012126-M,TOYOTA HILUX TRA 1a 21 X 26 1/2 2RN 1 9/16 X 1...,mayo 2025,1565.59,7.0,199.691429
1,RECESA,Allan,HOJAS,21012127-G,TOYOTA HILUX TRA 1a,mayo 2025,1246.20,6.0,185.446667
2,RECESA,Allan,HOJAS,21012229-A,TOYOTA HILUX 89 2WD TRA 1a S/J,mayo 2025,1605.60,7.0,204.795714
3,RECESA,Allan,HOJAS,21012229-B,TOYOTA HILUX 90 4WD TRA 1a C/J,mayo 2025,2106.84,9.0,209.011111
4,RECESA,Allan,HOJAS,21012323-B,TOYOTA HILUX DEL 1a,mayo 2025,842.48,4.0,188.055000
5,RECESA,Allan,HOJAS,21012424-G,SUZUKI CARRY APV TRA 1a C/1J,mayo 2025,1022.95,3.0,304.450000
6,RECESA,Allan,HOJAS,21012428-A,TOYOTA HILUX TURBO DIESEL D4D TRA 1a C/J 4X4,mayo 2025,330.63,1.0,295.210000
7,RECESA,Allan,HOJAS,21012431-M,TOYOTA TACOMA TRA 1a S/J 24X31,mayo 2025,541.56,2.0,241.770000
8,RECESA,Allan,HOJAS,21022127-K,TOYOTA HILUX TRA 2a,mayo 2025,1965.43,10.0,175.485000
9,RECESA,Allan,HOJAS,21022229-A,TOYOTA HILUX 90 4WD TRA 2a C/J,mayo 2025,2067.44,10.0,184.593000


## 5. Ordenar meses

In [7]:
meses_num = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "setiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

df_total_vendedores["Numero_Mes"] = (
    df_total_vendedores["Mes"]
    .str.extract(r"([a-zA-Záéíóúñ]+)")[0]
    .str.lower()
    .map(meses_num)
)

df_total_vendedores["Año"] = (
    df_total_vendedores["Mes"]
    .str.extract(r"(\d{4})")[0]
    .astype(int)
)

df_total_vendedores["Mes_Texto"] = df_total_vendedores["Mes"]

df_total_vendedores["Orden_Mes"] = (
    df_total_vendedores["Año"] * 100 +
    df_total_vendedores["Numero_Mes"]
)

df_total_vendedores = df_total_vendedores.sort_values(
    ["Orden_Mes", "Vendedor", "Categoria", "Codigo"]
).reset_index(drop=True)

df_total_vendedores.head(20)

,Empresa,Vendedor,Categoria,Codigo,Descripcion,Mes,Total,Cantidad,Precio_Promedio,Numero_Mes,Año,Mes_Texto,Orden_Mes
0,RECESA,Allan,HOJAS,21012126-M,TOYOTA HILUX TRA 1a 21 X 26 1/2 2RN 1 9/16 X 1...,mayo 2025,1565.59,7.0,199.691429,5,2025,mayo 2025,202505
1,RECESA,Allan,HOJAS,21012127-G,TOYOTA HILUX TRA 1a,mayo 2025,1246.20,6.0,185.446667,5,2025,mayo 2025,202505
2,RECESA,Allan,HOJAS,21012229-A,TOYOTA HILUX 89 2WD TRA 1a S/J,mayo 2025,1605.60,7.0,204.795714,5,2025,mayo 2025,202505
3,RECESA,Allan,HOJAS,21012229-B,TOYOTA HILUX 90 4WD TRA 1a C/J,mayo 2025,2106.84,9.0,209.011111,5,2025,mayo 2025,202505
4,RECESA,Allan,HOJAS,21012323-B,TOYOTA HILUX DEL 1a,mayo 2025,842.48,4.0,188.055000,5,2025,mayo 2025,202505
5,RECESA,Allan,HOJAS,21012424-G,SUZUKI CARRY APV TRA 1a C/1J,mayo 2025,1022.95,3.0,304.450000,5,2025,mayo 2025,202505
6,RECESA,Allan,HOJAS,21012428-A,TOYOTA HILUX TURBO DIESEL D4D TRA 1a C/J 4X4,mayo 2025,330.63,1.0,295.210000,5,2025,mayo 2025,202505
7,RECESA,Allan,HOJAS,21012431-M,TOYOTA TACOMA TRA 1a S/J 24X31,mayo 2025,541.56,2.0,241.770000,5,2025,mayo 2025,202505
8,RECESA,Allan,HOJAS,21022127-K,TOYOTA HILUX TRA 2a,mayo 2025,1965.43,10.0,175.485000,5,2025,mayo 2025,202505
9,RECESA,Allan,HOJAS,21022229-A,TOYOTA HILUX 90 4WD TRA 2a C/J,mayo 2025,2067.44,10.0,184.593000,5,2025,mayo 2025,202505


## 6. Resumen general por vendedor

In [8]:
resumen_vendedor = (
    df_total_vendedores
    .groupby("Vendedor", as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
)

resumen_vendedor["Precio_Promedio"] = (
    resumen_vendedor["Total"] /
    resumen_vendedor["Cantidad"]
).replace([np.inf, -np.inf], 0).fillna(0)

resumen_vendedor

,Vendedor,Total,Cantidad,Precio_Promedio
0,Allan,2374813.90,28946.5,82.041487
1,Fabiola,3071117.34,853812.0,3.596948


## 7. Resumen mensual por vendedor

In [9]:
resumen_mensual = (
    df_total_vendedores
    .groupby(["Orden_Mes", "Mes_Texto", "Vendedor"], as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
    .sort_values(["Orden_Mes", "Vendedor"])
)

resumen_mensual

,Orden_Mes,Mes_Texto,Vendedor,Total,Cantidad
0,202505,mayo 2025,Allan,134800.04,1071.5
1,202505,mayo 2025,Fabiola,955469.19,181385.0
2,202506,junio 2025,Allan,182507.56,2976.0
3,202506,junio 2025,Fabiola,108852.11,17235.0
4,202507,julio 2025,Allan,175718.68,1899.0
5,202507,julio 2025,Fabiola,591992.67,238130.0
6,202508,agosto 2025,Allan,219594.46,1486.0
7,202508,agosto 2025,Fabiola,80704.94,30934.0
8,202509,septiembre 2025,Allan,154163.40,1383.0
9,202509,septiembre 2025,Fabiola,229117.92,37852.0


## 8. Resumen por categoría y vendedor

In [10]:
resumen_categoria = (
    df_total_vendedores
    .groupby(["Vendedor", "Categoria"], as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
    .sort_values(["Vendedor", "Total"], ascending=[True, False])
)

resumen_categoria

,Vendedor,Categoria,Total,Cantidad
1,Allan,HOJAS,1370399.30,3958.0
6,Allan,TORNILLO CENTRO,396708.17,21684.5
5,Allan,RESORTAJE,310141.51,164.0
2,Allan,LAÑA,212624.95,2493.0
4,Allan,PERNO ESTRUCTURAL,42717.92,357.0
3,Allan,PERNO,41614.89,288.0
0,Allan,BALANCIN,607.16,2.0
12,Fabiola,TORNILLO CENTRO,1505538.30,707494.0
10,Fabiola,PERNO ESTRUCTURAL,1206737.78,22316.0
11,Fabiola,TORNILLERIA Y FERRETERIA,316418.93,122911.0


## 9. Top productos general

In [11]:
top_productos = (
    df_total_vendedores
    .groupby(["Codigo", "Descripcion"], as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
    .sort_values("Total", ascending=False)
)

top_productos.head(20)

,Codigo,Descripcion,Total,Cantidad
685,PSM,PERNO S/MUESTRA,1234813.28,22539.0
926,ROA32534212,TOR. HEX. A325 3/4 X 2.1/2,425953.20,40115.0
1246,TROA32534,TUERCA HEX. A325 3/4,171696.30,64062.0
702,RC-0522,RESORTE C. TOYOTA HILUX REVO 3H + 2 AUX (1 RB-...,160894.64,79.0
424,EL-2423,ELECTRODO 7018 1/8 (X LIBRA),95294.00,5800.0
1261,VA-3416,"ELECTRODO 6013 3/32"" (LIBRA)",73356.80,5731.0
1248,TROA32578,TUERCA HEX. A325 7/8,65138.75,11992.0
23,21012229-B,TOYOTA HILUX 90 4WD TRA 1a C/J,57733.16,241.0
1242,TROA3251,"TUERCA HEX. A325 1""",55342.37,7921.0
46,21022229-A,TOYOTA HILUX 90 4WD TRA 2a C/J,53663.85,253.0


## 10. Top productos por vendedor

In [12]:
top_productos_vendedor = (
    df_total_vendedores
    .groupby(["Vendedor", "Codigo", "Descripcion"], as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
    .sort_values(["Vendedor", "Total"], ascending=[True, False])
)

top_productos_vendedor.head(30)

,Vendedor,Codigo,Descripcion,Total,Cantidad
589,Allan,RC-0522,RESORTE C. TOYOTA HILUX REVO 3H + 2 AUX (1 RB-...,160894.64,79.0
22,Allan,21012229-B,TOYOTA HILUX 90 4WD TRA 1a C/J,57006.69,238.0
45,Allan,21022229-A,TOYOTA HILUX 90 4WD TRA 2a C/J,53663.85,253.0
40,Allan,21022127-K,TOYOTA HILUX TRA 2a,43653.05,216.0
330,Allan,90023131-A,NEWAY TS-5600 2a 30 5X30 5 5X1 C/CHICHE,42600.00,30.0
83,Allan,22022828-A,TOYOTA HILUX REVO TRA. 2a. 2 1/4 X 3/8 C/J,39306.50,92.0
99,Allan,25HL1417-A,TOYOTA HILUX AUXILIAR 5/8 CON ABRAZADERA,34740.78,133.0
328,Allan,90013131-C,NEWAY TRA-3224-01,34000.00,24.0
86,Allan,22032525-M,TOYOTA HILUX REVO TRA. 3a.,31824.64,77.0
378,Allan,LCC1185181512,LANA CUADRADA 1 1/8 X 5 1/8 X 15 1/2 CON TUERCA,31460.00,143.0


## 11. Validaciones rápidas

In [14]:
print("VENTAS TOTALES:", round(df_total_vendedores["Total"].sum(), 2))
print("UNIDADES TOTALES:", round(df_total_vendedores["Cantidad"].sum(), 2))

print("\nMESES DETECTADOS:")
print(
    df_total_vendedores[["Mes_Texto", "Orden_Mes"]]
    .drop_duplicates()
    .sort_values("Orden_Mes")
    .to_string(index=False)
)

print("\nVENDEDORES DETECTADOS:")
print(df_total_vendedores["Vendedor"].unique())

VENTAS TOTALES: 5445931.24
UNIDADES TOTALES: 882758.5

MESES DETECTADOS:
      Mes_Texto  Orden_Mes
      mayo 2025     202505
     junio 2025     202506
     julio 2025     202507
    agosto 2025     202508
septiembre 2025     202509
   octubre 2025     202510
 noviembre 2025     202511
 diciembre 2025     202512
     enero 2026     202601
   febrero 2026     202602
     marzo 2026     202603
     abril 2026     202604
      mayo 2026     202605
     junio 2026     202606

VENDEDORES DETECTADOS:
['Allan' 'Fabiola']


## 12. Exportar Excel final

In [15]:
with pd.ExcelWriter(
    "Base_Maestra_Vendedores_RECESA.xlsx",
    engine="openpyxl"
) as writer:

    df_total_vendedores.to_excel(
        writer,
        sheet_name="Base Maestra",
        index=False
    )

    resumen_vendedor.to_excel(
        writer,
        sheet_name="Resumen Vendedor",
        index=False
    )

    resumen_mensual.to_excel(
        writer,
        sheet_name="Resumen Mensual",
        index=False
    )

    resumen_categoria.to_excel(
        writer,
        sheet_name="Resumen Categoria",
        index=False
    )

    top_productos.to_excel(
        writer,
        sheet_name="Top Productos",
        index=False
    )

    top_productos_vendedor.to_excel(
        writer,
        sheet_name="Top por Vendedor",
        index=False
    )

print("Archivo generado correctamente: Base_Maestra_Vendedores_RECESA.xlsx")

Archivo generado correctamente: Base_Maestra_Vendedores_RECESA.xlsx
